In [1]:
import subprocess
import sys

def instalar_si_falta(paquete):
    try:
        __import__(paquete.replace("-", "_"))
        print(f"{paquete} ya está instalado")
    except ImportError:
        print(f"Instalando {paquete}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", paquete])

# Lista de paquetes a verificar
paquetes = ["langchain", "langchain_groq", "langchain_community", "ddgs", "dotenv"]

for pkg in paquetes:
    instalar_si_falta(pkg)

print("Todos los paquetes están listos!")

langchain ya está instalado
langchain_groq ya está instalado
langchain_community ya está instalado
ddgs ya está instalado
dotenv ya está instalado
Todos los paquetes están listos!


In [2]:
import os
from dotenv import load_dotenv
from langchain_community.tools import DuckDuckGoSearchResults

load_dotenv()

# Verificar que la API key existe
groq_api_key = os.getenv("APY_KEY_GROQ")
if not groq_api_key:
    raise ValueError("❌ GROQ_API_KEY no encontrada en .env")
else:
    print("✅ API Key de Groq cargada correctamente")

# Ver los primeros caracteres de la key (seguro)

✅ API Key de Groq cargada correctamente


In [3]:
from langchain_groq import ChatGroq

# Elige el modelo que prefieras
MODELO = "llama-3.3-70b-versatile"  # Potente y gratuito

llm = ChatGroq(
    groq_api_key=groq_api_key,
    model=MODELO,
    temperature=0.2,           # Baja temperatura = más consistente
    max_retries=2,             # Reintentar si falla
    timeout=60,                # Timeout en segundos
    verbose=True               # Mostrar logs detallados
)

print(f"✅ LLM de Groq inicializado")
print(f"   Modelo: {MODELO}")
print(f"   Temperatura: 0.2")

✅ LLM de Groq inicializado
   Modelo: llama-3.3-70b-versatile
   Temperatura: 0.2


In [4]:
# Celda 4: Prueba simple de Groq
from langchain_core.messages import HumanMessage, SystemMessage

mensajes = [
    SystemMessage(content="Eres un asistente útil. Responde de manera concisa."),
    HumanMessage(content="¿Cuál es la capital de Argentina?")
]

respuesta = llm.invoke(mensajes)
print("🤖 Respuesta de Groq:")
print("-" * 40)
print(respuesta.content)
print("-" * 40)
print(f"\n✅ Tokens usados: {respuesta.response_metadata.get('token_usage', {})}")

🤖 Respuesta de Groq:
----------------------------------------
La capital de Argentina es Buenos Aires.
----------------------------------------

✅ Tokens usados: {'completion_tokens': 9, 'prompt_tokens': 60, 'total_tokens': 69, 'completion_time': 0.006886983, 'completion_tokens_details': None, 'prompt_time': 0.014774499, 'prompt_tokens_details': None, 'queue_time': 0.157920267, 'total_time': 0.021661482}


In [ ]:
# Celda: Clase Agent_Search corregida
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.tools import DuckDuckGoSearchResults
from typing import List, Dict, Any,TypedDict, Annotated, List, Dict, Any, Literal

class Agent_Search:
    def __init__(self, llm):
        self.llm = llm
        self.searching = DuckDuckGoSearchResults(
            output_format="list",
            num_results=3,  # Nota: es 'num_results', no 'max_results'
            backend="text"
        )
        print("✅ Agente de búsqueda inicializado")
        
        # Manejar diferentes formas de obtener el nombre del modelo
        modelo = getattr(llm, 'model_name', getattr(llm, 'model', 'Desconocido'))
        temperatura = getattr(llm, 'temperature', 'N/A')
        print(f"🧠 LLM: modelo {modelo}, temperatura {temperatura}")
        
    def buscar(self, query: str) -> Dict[str, Any]:
        """
        Realiza una búsqueda y devuelve resultados formateados.
        """
        print(f"\n🔍 BUSCANDO: '{query}'")
        
        try:
            # CORRECCIÓN 1: Usar invoke() en lugar de run()
            output = self.searching.invoke(query)
            
            # CORRECCIÓN 2: Verificar el tipo de output
            if isinstance(output, str):
                # Si es string, no podemos formatearlo como lista
                return {
                    "exito": True,
                    "Query": query,
                    "cantidad": 1,
                    "resultados": [{"contenido": output, "fuente": "DuckDuckGo"}],
                    "texto_completo": output,
                    "raw_output": output
                }
            
            # Si es lista, procesar normalmente
            output_formatted = self._formatear_output(output)
            
            return {
                "exito": True,
                "Query": query,
                "cantidad": len(output_formatted),
                "resultados": output_formatted,
                "texto_completo": self._a_texto(output_formatted),
                # CORRECCIÓN 3: DuckDuckGo NO tiene metadata de tokens
                # Solo el LLM tiene tokens, no la búsqueda
                "Tokens_usados": {}  # DuckDuckGo no tiene esta info
            }
            
        except Exception as e:
            print(f"❌ Error en búsqueda: {e}")
            return {
                "exito": False,
                "Query": query,
                "error": str(e),
                "resultados": []
            }
    
    def _formatear_output(self, resultados_raw: list) -> list:
        """Formatea los resultados de DuckDuckGo"""
        resultados = []
        
        for r in resultados_raw:
            resultado = {
                "titulo": r.get('title', 'Sin título'),
                "url": r.get('link', r.get('url', 'Sin URL')),
                "contenido": r.get('snippet', r.get('content', 'Sin contenido')),
                "fuente": "DuckDuckGo"
            }
            resultados.append(resultado)
            print(f"  📄 {resultado['titulo'][:60]}...")
            
        return resultados
    
    def _a_texto(self, resultados: list) -> str:
        """Convierte resultados a texto plano para usar en prompts"""
        texto = "RESULTADOS DE BÚSQUEDA:\n\n"
        
        for i, r in enumerate(resultados, 1):
            texto += f"--- FUENTE {i} ---\n"
            texto += f"Título: {r['titulo']}\n"
            texto += f"URL: {r['url']}\n"
            texto += f"Contenido:\n{r['contenido']}\n\n"
        
        return texto
    
    def resumir_query(self, query: str, resultado: Dict) -> str:
        """
        Resume los resultados usando el LLM.
        CORRECCIÓN 4: Usar invoke() en lugar de run()
        """
        if not resultado.get("exito", False):
            return f"No se pudo obtener resultados para resumir. Error: {resultado.get('error', 'Desconocido')}"
        
        prompt = ChatPromptTemplate.from_template("""Eres un asistente de investigación. Basado en la siguiente información de búsqueda,
        responde la pregunta del usuario de manera clara y concisa.
        
        PREGUNTA: {query}
        
        INFORMACIÓN ENCONTRADA:
        {informacion}
        
        INSTRUCCIONES:
        1. Responde directamente a la pregunta usando SOLO la información proporcionada
        2. Si la información es insuficiente, indícalo claramente
        3. Cita las fuentes cuando sea relevante
        4. Sé conciso (máximo 3-4 párrafos)
        
        RESPUESTA:
        """)
        
        cadena = prompt | self.llm
        # CORRECCIÓN 5: Usar invoke() con diccionario, no run()
        respuesta = cadena.invoke({"query": query, "informacion": resultado["texto_completo"]})
        
        return respuesta.content
    
    def resumir_query_con_tokens(self, query: str, resultado: Dict) -> Dict:
        """
        Versión que también devuelve información de tokens del LLM.
        """
        if not resultado.get("exito", False):
            return {
                "contenido": f"Error: {resultado.get('error', 'Desconocido')}",
                "tokens": {},
                "modelo": None
            }
        
        prompt = ChatPromptTemplate.from_template("""Eres un asistente de investigación. Basado en la siguiente información de búsqueda,
        responde la pregunta del usuario de manera clara y concisa.
        
        PREGUNTA: {query}
        
        INFORMACIÓN ENCONTRADA:
        {informacion}
        
        INSTRUCCIONES:
        1. Responde directamente a la pregunta usando SOLO la información proporcionada
        2. Si la información es insuficiente, indícalo claramente
        3. Cita las fuentes cuando sea relevante
        4. Sé conciso (máximo 3-4 párrafos)
        
        RESPUESTA:
        """)
        
        cadena = prompt | self.llm
        respuesta = cadena.invoke({"query": query, "informacion": resultado["texto_completo"]})
        
        # Extraer metadata de tokens
        metadata = respuesta.response_metadata
        token_usage = metadata.get('token_usage', {})
        
        return {
            "contenido": respuesta.content,
            "tokens": {
                "prompt_tokens": token_usage.get('prompt_tokens', 'N/A'),
                "completion_tokens": token_usage.get('completion_tokens', 'N/A'),
                "total_tokens": token_usage.get('total_tokens', 'N/A')
            },
            "modelo": metadata.get('model_name', 'Desconocido'),
            "respuesta_raw": respuesta  # Para depuración si es necesario
        }

In [6]:
# Celda 5: Probar búsqueda simple
agente=Agent_Search(llm)

query = input(("🔎 Ingresa tu consulta de búsqueda: "))
resultado = agente.buscar(query)

print(f"\n📊 RESULTADO:")
print(f"  ✅ Éxito: {resultado['exito']}")
print(f"  📝 Query: {resultado['Query']}")
print(f"  🔢 Resultados encontrados: {resultado['cantidad']}")
resumen = agente.resumir_query_con_tokens(query, resultado)
print(f"  📄 Resumen: {resumen['contenido']}")
if resumen['tokens']:
        print(f"  📥 Prompt tokens:     {resumen['tokens']['prompt_tokens']}")
        print(f"  📤 Completion tokens: {resumen['tokens']['completion_tokens']}")
        print(f"  🔢 Total tokens:      {resumen['tokens']['total_tokens']}")
        print(f"  🧠 Modelo:            {resumen['modelo']}")

✅ Agente de búsqueda inicializado
🧠 LLM: modelo llama-3.3-70b-versatile, temperatura 0.2

🔍 BUSCANDO: 'Pais mas cercano a Argentina'
  📄 Countries Near Argentina 2026 - Border Nations, Distance in ...
  📄 Top 5 Playas del Sur de Brasil Más Cercanas a la Argentina p...
  📄 Qué país tiene más fronteras con otros países: dos limitan ....

📊 RESULTADO:
  ✅ Éxito: True
  📝 Query: Pais mas cercano a Argentina
  🔢 Resultados encontrados: 3
  📄 Resumen: La pregunta del usuario es sobre el país más cercano a Argentina. Desafortunadamente, la información proporcionada no es suficiente para determinar con precisión cuál es el país más cercano a Argentina. 

Según la FUENTE 1, se menciona que hay países que comparten una frontera terrestre con Argentina, pero no se especifica cuál es el más cercano. La FUENTE 2 habla sobre playas en el sur de Brasil que están cerca de la frontera con Argentina, lo que sugiere que Brasil podría ser un país cercano, pero no se proporciona información sobre la distan

In [15]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import AnyMessage, HumanMessage, AIMessage

# Celda 3: Definir el estado del sistema
class InvestigacionState(TypedDict):
    """
    Estado compartido entre todos los agentes.
    LangGraph pasa este objeto de nodo a nodo.
    """
    # Historial de mensajes (se acumula automáticamente con operator.add)
    messages: Annotated[List[AnyMessage], operator.add]
    
    # Información estática
    pregunta_original: str
    
    # Contador de iteraciones
    iteracion: int
    
    # Información acumulada de búsquedas (lista de diccionarios)
    info_acumulada: List[Dict[str, Any]]
    
    # Decisiones del crítico
    decision_critico: str
    razon_critico: str
    
    # Próxima consulta (si el crítico pide más)
    consulta_siguiente: str
    
    # Informe final
    informe_final: str
    
    # Metadata adicional
    tokens_totales: int

# Estado inicial por defecto
def crear_estado_inicial(pregunta: str) -> InvestigacionState:
    """Crea el estado inicial para una nueva investigación"""
    return {
        "messages": [HumanMessage(content=pregunta)],
        "pregunta_original": pregunta,
        "iteracion": 0,
        "info_acumulada": [],
        "decision_critico": "",
        "razon_critico": "",
        "consulta_siguiente": pregunta,
        "informe_final": "",
        "tokens_totales": 0
    }

print("✅ Estado definido")

✅ Estado definido


In [8]:
class Agent_Critico:
    """Evalúa si la información es suficiente o necesita más búsqueda"""
    
    def __init__(self, llm):
        self.llm = llm
        self.prompt = ChatPromptTemplate.from_template("""
Eres un evaluador crítico de información. Tu tarea es determinar si la información encontrada
es SUFICIENTE para responder la pregunta original o si se necesita MÁS BÚSQUEDA.

Pregunta original: {pregunta}
Iteración actual: {iteracion}
Información acumulada hasta ahora:
{informacion}

Evalúa según estos criterios:
- ¿La información responde directamente la pregunta?
- ¿Hay datos concretos o solo opiniones vagas?
- ¿Se necesitan múltiples perspectivas?
- ¿La información está actualizada?

Responde en este formato EXACTO (sin texto adicional):
DECISIÓN: [SUFICIENTE o NECESITA_MAS]
RAZÓN: (explica en 1 línea por qué)
CONSULTA_ADICIONAL: (si es NECESITA_MAS, escribe la nueva consulta; si es SUFICIENTE, escribe "NINGUNA")
""")
        print("✅ Agente Crítico inicializado")
    
    def evaluar(self, pregunta: str, informacion: str, iteracion: int) -> dict:
        print(f"\n⚖️ CRÍTICO: Evaluando información (Iteración {iteracion})...")
        
        cadena = self.prompt | self.llm
        respuesta = cadena.invoke({
            "pregunta": pregunta,
            "informacion": informacion,
            "iteracion": iteracion
        })
        
        # Parsear respuesta
        contenido = respuesta.content
        lineas = contenido.strip().split('\n')
        
        decision = "SUFICIENTE"
        razon = "No se pudo determinar"
        consulta = None
        
        for linea in lineas:
            if linea.startswith("DECISIÓN:"):
                decision = linea.replace("DECISIÓN:", "").strip()
            elif linea.startswith("RAZÓN:"):
                razon = linea.replace("RAZÓN:", "").strip()
            elif linea.startswith("CONSULTA_ADICIONAL:"):
                consulta_raw = linea.replace("CONSULTA_ADICIONAL:", "").strip()
                consulta = consulta_raw if consulta_raw != "NINGUNA" else None
        
        print(f"  📊 Decisión: {decision}")
        print(f"  💡 Razón: {razon}")
        
        return {
            "suficiente": decision == "SUFICIENTE",
            "decision": decision,
            "razon": razon,
            "consulta_adicional": consulta
        }
agente_critico = Agent_Critico(llm)

✅ Agente Crítico inicializado


In [9]:
print("="*60)
print("TEST 1: INFORMACIÓN COMPLETA (debería decir SUFICIENTE)")
print("="*60)

pregunta_test = "¿Cuál es la capital de Francia?"

informacion_completa = """
RESULTADOS DE BÚSQUEDA:

--- FUENTE 1 ---
Título: Paris, la capital de Francia
URL: https://ejemplo.com/paris-francia
Contenido:
París es la capital de Francia desde el siglo X. Es la ciudad más poblada del país 
y alberga las principales instituciones gubernamentales. La Torre Eiffel es su 
monumento más emblemático.

--- FUENTE 2 ---
Título: Datos sobre París
URL: https://ejemplo.com/datos-paris
Contenido:
París es la capital y ciudad más grande de Francia. Está situada a orillas del río Sena.
"""

resultado = agente_critico.evaluar(pregunta_test, informacion_completa, iteracion=1)
print(f"\n✅ ¿Suficiente? {resultado['suficiente']}")
print(f"📊 Decisión final: {resultado['decision']}")

TEST 1: INFORMACIÓN COMPLETA (debería decir SUFICIENTE)

⚖️ CRÍTICO: Evaluando información (Iteración 1)...
  📊 Decisión: SUFICIENTE
  💡 Razón: La información proporcionada por ambas fuentes confirma directamente que París es la capital de Francia.

✅ ¿Suficiente? True
📊 Decisión final: SUFICIENTE


In [10]:
# Celda 6: Definir Agente Redactor
class Agent_Redactor:
    """Sintetiza la información en un informe final estructurado"""
    
    def __init__(self, llm):
        self.llm = llm
        self.prompt = ChatPromptTemplate.from_template("""
Eres un redactor profesional de informes. Tu tarea es sintetizar la información recopilada
en un informe estructurado, claro y accionable.

Pregunta original: {pregunta}
Información recopilada (múltiples fuentes):
{informacion}

Genera un informe con la siguiente estructura:

# INFORME: {pregunta}

## 📌 RESUMEN EJECUTIVO
(2-3 líneas con lo más importante)

## 🔍 HALLAZGOS PRINCIPALES
- Hallazgo 1 con evidencia de las fuentes
- Hallazgo 2 con evidencia de las fuentes
- Hallazgo 3 con evidencia de las fuentes

## 📊 ANÁLISIS CRÍTICO
(Interpretación de los hallazgos, puntos de acuerdo/desacuerdo entre fuentes)

## 💡 CONCLUSIONES Y RECOMENDACIONES
(Respuesta directa a la pregunta y recomendaciones si aplica)

## 📚 FUENTES CONSULTADAS
(Lista de URLs mencionadas en la información)

El informe debe ser profesional, objetivo y basado ESTRICTAMENTE en la información proporcionada.
""")
        print("✅ Agente Redactor inicializado")
    
    def redactar(self, pregunta: str, informacion: str) -> dict:
        """Redacta un informe estructurado"""
        print(f"\n📝 REDACTOR: Generando informe final...")
        
        cadena = self.prompt | self.llm
        respuesta = cadena.invoke({
            "pregunta": pregunta,
            "informacion": informacion
        })
        
        # Extraer metadata de tokens
        metadata = respuesta.response_metadata
        token_usage = metadata.get('token_usage', {})
        
        print(f"  ✅ Informe completado")
        print(f"  📊 Tokens usados: {token_usage.get('total_tokens', 'N/A')}")
        
        return {
            "contenido": respuesta.content,
            "tokens": token_usage,
            "modelo": metadata.get('model_name', 'Desconocido')
        }
    
    def redactar_simple(self, pregunta: str, informacion: str) -> str:
        """Versión simple que solo devuelve el texto"""
        resultado = self.redactar(pregunta, informacion)
        return resultado["contenido"]

# Crear instancia
redactor = Agent_Redactor(llm)

✅ Agente Redactor inicializado


In [11]:
# Celda 7: Test 1 - Información simple
print("="*60)
print("TEST REDACTOR 1: INFORME SOBRE CAPITAL DE FRANCIA")
print("="*60)

pregunta_simple = "¿Cuál es la capital de Francia?"

info_simple = """
RESULTADOS DE BÚSQUEDA:

--- FUENTE 1 ---
Título: París, la capital de Francia
URL: https://es.wikipedia.org/wiki/Par%C3%ADs
Contenido:
París es la capital de Francia y su ciudad más poblada. Está situada a orillas del río Sena.
Es conocida como "La Ciudad de la Luz" y alberga monumentos como la Torre Eiffel, 
el Museo del Louvre y la Catedral de Notre-Dame.
"""

informe = redactor.redactar(pregunta_simple, info_simple)
print("\n" + "="*60)
print("INFORME GENERADO")
print("="*60)
print(informe["contenido"])

TEST REDACTOR 1: INFORME SOBRE CAPITAL DE FRANCIA

📝 REDACTOR: Generando informe final...
  ✅ Informe completado
  📊 Tokens usados: 883

INFORME GENERADO
# INFORME: ¿Cuál es la capital de Francia?

## 📌 RESUMEN EJECUTIVO
La capital de Francia es París, conocida como "La Ciudad de la Luz". Esta ciudad es la más poblada del país y alberga varios monumentos icónicos. La información recopilada confirma que París es el centro político y cultural de Francia.

## 🔍 HALLAZGOS PRINCIPALES
- **Ubicación y características**: París está situada a orillas del río Sena, lo que la convierte en un punto geográfico importante.
- **Monumentos y atracciones**: La ciudad es famosa por albergar monumentos como la Torre Eiffel, el Museo del Louvre y la Catedral de Notre-Dame, lo que la convierte en un destino turístico popular.
- **Importancia cultural y política**: París es conocida como "La Ciudad de la Luz" y es la ciudad más poblada de Francia, lo que subraya su importancia cultural y política en el paí